# Workflows de Documentos con n8n y Claude

Implementación en Python de los patrones de procesamiento de documentos
que se despliegan como workflows visuales en n8n.

**Casos de uso:**
1. Extracción de datos de facturas
2. Clasificación de contratos
3. Resumen de documentos largos
4. Enriquecimiento de leads desde formularios

In [ ]:
import anthropic
import json
from datetime import datetime

client = anthropic.Anthropic()
print("Cliente Anthropic listo")

## Workflow 1: Extracción de datos de facturas

En n8n: Email Trigger → Extract from File (PDF) → HTTP Request Claude → Google Sheets

In [ ]:
# Simula el texto extraído del PDF por el nodo "Extract from File" de n8n
FACTURA_EJEMPLO = """
FACTURA N.º: FAC-2026-0342
Fecha de emisión: 15/04/2026
Fecha de vencimiento: 15/05/2026

PROVEEDOR:
TechSupply Solutions SL
NIF: B-87654321
Calle Gran Vía 45, 28013 Madrid

CLIENTE:
Mi Empresa SA
NIF: A-12345678

CONCEPTOS:
- Licencias software ERP (10 usuarios x 12 meses)  2.400,00 €
- Soporte técnico premium anual                      600,00 €
- Formación inicial (2 días)                         800,00 €

SUBTOTAL: 3.800,00 €
IVA (21%): 798,00 €
TOTAL: 4.598,00 €

Método de pago: Transferencia bancaria
IBAN: ES76 0049 1234 5612 3456 7890
N.º pedido: PO-2026-112
"""

print("Texto de factura listo para procesar")
print(f"Longitud: {len(FACTURA_EJEMPLO)} caracteres")

In [ ]:
def extraer_datos_factura(texto_pdf: str) -> dict:
    """Equivalente al nodo Code que llama a Claude API en el workflow de facturas."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=600,
        messages=[{
            "role": "user",
            "content": f"""Extrae los datos de esta factura. Responde SOLO con JSON válido, sin markdown.

FACTURA:
{texto_pdf[:3000]}

JSON requerido:
{{
  "numero_factura": "...",
  "fecha_emision": "YYYY-MM-DD",
  "fecha_vencimiento": "YYYY-MM-DD",
  "proveedor": {{"nombre": "...", "nif": "...", "direccion": "..."}},
  "cliente": {{"nombre": "...", "nif": "..."}},
  "lineas": [
    {{"descripcion": "...", "total": 0.00}}
  ],
  "subtotal": 0.00,
  "iva_pct": 21,
  "iva_importe": 0.00,
  "total": 0.00,
  "moneda": "EUR",
  "numero_pedido": "...",
  "metodo_pago": "...",
  "confianza": "alta|media|baja"
}}"""
        }]
    )
    
    datos = json.loads(response.content[0].text)
    datos["procesado_en"] = datetime.now().isoformat()
    datos["requiere_revision"] = datos["confianza"] != "alta" or datos["total"] > 5000
    
    return datos

factura = extraer_datos_factura(FACTURA_EJEMPLO)
print(json.dumps(factura, indent=2, ensure_ascii=False))

In [ ]:
# Simula el nodo IF + Google Sheets append
def procesar_resultado_factura(datos: dict):
    """Equivalente al nodo IF y las acciones posteriores en n8n."""
    print("\n" + "="*50)
    print("DECISIÓN (nodo IF en n8n):")
    
    if datos["requiere_revision"]:
        print(f"  → Rama VERDADERA: Slack con alerta de revisión")
        print(f"  Motivo: confianza={datos['confianza']}, total={datos['total']}€")
        print(f"  [SLACK #contabilidad] ⚠️ Factura {datos['numero_factura']} requiere revisión")
    else:
        print(f"  → Rama FALSA: guardar directamente en Google Sheets")
        fila = [
            datos['numero_factura'],
            datos['fecha_emision'],
            datos['proveedor']['nombre'],
            datos['total'],
            datos['confianza'],
            datos['procesado_en']
        ]
        print(f"  [SHEETS] Fila añadida: {fila}")

procesar_resultado_factura(factura)

## Workflow 2: Clasificación de contratos

In [ ]:
CONTRATO_EJEMPLO = """
CONTRATO DE PRESTACIÓN DE SERVICIOS

De una parte, TECNOLOGÍA AVANZADA SL (en adelante "el Proveedor"), y
de otra, EMPRESA CLIENTE SA (en adelante "el Cliente").

Fecha de inicio: 1 de mayo de 2026
Duración: 12 meses con renovación automática salvo preaviso de 30 días.

Objeto: El Proveedor prestará servicios de desarrollo de software a medida
para el sistema de gestión interno del Cliente.

Valor del contrato: 48.000 € anuales, facturado mensualmente.

Cláusula de confidencialidad: Las partes se obligan a mantener secreto
sobre toda información confidencial intercambiada durante la vigencia del contrato.

Propiedad intelectual: El software desarrollado será propiedad del Cliente
una vez recibido el pago completo.

Penalización por incumplimiento: El retraso en la entrega superiora 7 días
conlleva una penalización del 1% del valor mensual por día de retraso.
"""

def clasificar_contrato(texto: str) -> dict:
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        messages=[{
            "role": "user",
            "content": f"""Analiza este contrato. JSON sin markdown:
{{
  "tipo": "prestacion_servicios|compraventa|arrendamiento|laboral|confidencialidad|otro",
  "partes": ["Empresa A", "Empresa B"],
  "fecha_inicio": "YYYY-MM-DD o null",
  "fecha_fin": "YYYY-MM-DD o null",
  "valor_contrato_eur": 0,
  "renovacion_automatica": true,
  "clausulas_criticas": ["lista de 1-3 cláusulas importantes"],
  "carpeta_archivo": "Proveedores|Clientes|RRHH|Legal|Otro"
}}

CONTRATO:
{texto[:4000]}"""
        }]
    )
    
    clasificacion = json.loads(response.content[0].text)
    año = datetime.now().year
    clasificacion["ruta_drive"] = f"Contratos/{clasificacion['tipo']}/{año}"
    clasificacion["nombre_archivo"] = f"{clasificacion['partes'][0]}_{clasificacion['fecha_inicio'] or 'sin_fecha'}.pdf"
    
    return clasificacion

contrato = clasificar_contrato(CONTRATO_EJEMPLO)
print(json.dumps(contrato, indent=2, ensure_ascii=False))

## Workflow 3: Resumen de documentos largos (chunking)

In [ ]:
# Simular un documento largo (informe anual, etc.)
DOCUMENTO_LARGO = """
INFORME ANUAL DE SOSTENIBILIDAD 2025

CAPÍTULO 1: RESUMEN EJECUTIVO
Durante 2025 hemos avanzado significativamente en nuestros objetivos de sostenibilidad.
Reducimos emisiones de CO2 en un 23% respecto al año anterior. Implementamos paneles
solares en 3 centros logísticos que ahora cubren el 40% del consumo eléctrico.
La cadena de suministro sostenible alcanzó al 67% de proveedores estratégicos.

CAPÍTULO 2: IMPACTO AMBIENTAL
Huella de carbono: 12.450 toneladas CO2eq (vs 16.200 en 2024)
Consumo de agua: reducido en 31% gracias al sistema de reciclaje instalado en planta.
Residuos: 89% de los residuos industriales se reciclan o reutilizan.
Packaging: transición al 100% de materiales reciclados completada en Q3 2025.

CAPÍTULO 3: IMPACTO SOCIAL
Empleados: 2.340 personas, +12% vs 2024. Brecha salarial de género: 3.2% (objetivo: 0%).
Formación: media de 42 horas de formación por empleado al año.
Accidentalidad: índice de frecuencia de 2.1, por debajo de la media sectorial (3.8).
Comunidad: 180.000€ invertidos en proyectos sociales locales.

CAPÍTULO 4: GOBERNANZA
Consejo de administración: 45% mujeres. Comité de sostenibilidad creado en enero 2025.
Auditorías externas: certificación ISO 14001 y SA8000 renovadas.
Código ético: formación obligatoria completada por el 98% de la plantilla.
Proveedores evaluados en criterios ESG: 234 de 310 proveedores activos.

CAPÍTULO 5: OBJETIVOS 2026
- Reducir emisiones un 30% adicional (carbono neutro en 2028)
- Eliminar brecha salarial de género
- Alcanzar 100% de proveedores estratégicos con evaluación ESG
- Lanzar programa de economía circular con 5 socios industriales
- Invertir 250.000€ en comunidad local
""" * 3  # Triplicar para simular documento realmente largo

print(f"Documento de {len(DOCUMENTO_LARGO)} caracteres")
print(f"Chunks necesarios (8000 chars): {len(DOCUMENTO_LARGO) // 8000 + 1}")

In [ ]:
def resumir_documento_largo(texto: str, max_chunk_chars: int = 8000) -> dict:
    """Patrón map-reduce para documentos que no caben en un solo contexto."""
    # Dividir en chunks
    chunks = [texto[i:i+max_chunk_chars] for i in range(0, len(texto), max_chunk_chars)]
    print(f"Dividido en {len(chunks)} partes")
    
    # Fase MAP: resumir cada chunk
    resumenes = []
    for idx, chunk in enumerate(chunks):
        print(f"  Resumiendo parte {idx+1}/{len(chunks)}...")
        resp = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=300,
            messages=[{
                "role": "user",
                "content": f"Resume este fragmento ({idx+1}/{len(chunks)}) en 3-4 bullets:\n\n{chunk}"
            }]
        )
        resumenes.append(resp.content[0].text)
    
    # Fase REDUCE: síntesis final
    print("  Generando síntesis final...")
    resp_final = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=500,
        messages=[{
            "role": "user",
            "content": f"Sintetiza estos resúmenes parciales en un resumen ejecutivo de máximo 200 palabras:\n\n{'---'.join(resumenes)}"
        }]
    )
    
    return {
        "resumen_ejecutivo": resp_final.content[0].text,
        "num_partes": len(chunks),
        "chars_totales": len(texto)
    }

resultado = resumir_documento_largo(DOCUMENTO_LARGO)
print("\n" + "="*50)
print("RESUMEN EJECUTIVO FINAL:")
print("="*50)
print(resultado["resumen_ejecutivo"])
print(f"\n(Procesadas {resultado['num_partes']} partes, {resultado['chars_totales']:,} caracteres)")

## Workflow 4: Enriquecimiento de leads desde formularios

In [ ]:
# Simula el webhook trigger de n8n recibiendo datos de un formulario web
LEADS_FORMULARIO = [
    {
        "nombre": "Carlos Martínez",
        "email": "carlos@acmecorp.com",
        "empresa": "ACME Corp",
        "cargo": "Director de Operaciones",
        "mensaje": "Buscamos automatizar el procesamiento de 500+ facturas mensuales de proveedores internacionales."
    },
    {
        "nombre": "Ana López",
        "email": "ana@startup.io",
        "empresa": "StartupIO",
        "cargo": "Fundadora",
        "mensaje": "Soy una startup de 5 personas, me interesa el plan básico para revisar contratos con clientes."
    }
]

def enriquecer_lead(datos_formulario: dict) -> dict:
    """Equivalente al nodo Code que enriquece leads antes de insertarlos en el CRM."""
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=300,
        messages=[{
            "role": "user",
            "content": f"""Enriquece este lead. JSON sin markdown:
{{
  "industria_detectada": "tecnología|retail|legal|salud|educacion|otro",
  "tamaño_empresa_estimado": "1-10|11-50|51-200|201-1000|1000+",
  "urgencia": "alta|media|baja",
  "caso_uso_principal": "descripción breve",
  "siguiente_accion": "llamada_inmediata|email_nurturing|demo_producto|descalificar",
  "score_lead": 0
}}

Nombre: {datos_formulario['nombre']}
Email: {datos_formulario['email']}
Empresa: {datos_formulario.get('empresa', 'N/A')}
Cargo: {datos_formulario.get('cargo', 'N/A')}
Mensaje: {datos_formulario.get('mensaje', 'N/A')}"""
        }]
    )
    
    enriquecimiento = json.loads(response.content[0].text)
    return {
        **datos_formulario,
        **enriquecimiento,
        "procesado_en": datetime.now().isoformat()
    }

print("Procesando leads del formulario web:")
for lead_raw in LEADS_FORMULARIO:
    lead = enriquecer_lead(lead_raw)
    print(f"\n{lead['nombre']} ({lead['empresa']})")
    print(f"  Score: {lead['score_lead']}/100 | Urgencia: {lead['urgencia']}")
    print(f"  Siguiente acción: {lead['siguiente_accion']}")
    print(f"  Caso de uso: {lead['caso_uso_principal']}")

## Resumen de patrones

```
FLUJO EN n8n                     EQUIVALENTE PYTHON
─────────────────                ─────────────────────
Email Trigger              →     dict con datos del email
Extract from File (PDF)    →     texto = leer_pdf(ruta)
HTTP Request → Claude      →     client.messages.create()
Code Node (parsear JSON)   →     json.loads(response.content[0].text)
IF Node (revisión manual)  →     if datos["requiere_revision"]
Google Sheets append       →     print(f"[SHEETS] {fila}")
Webhook Trigger            →     datos = request.json()
```

El patrón **map-reduce** (`resumir_documento_largo`) es clave para documentos
que superan el contexto de un solo prompt. En n8n se implementa con un Loop
Over Items node + HTTP Request dentro del loop.